# **Asphalt Pavement Condition Classification**

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Conv1D, Dense, Flatten, Dropout, MaxPooling1D, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import keras_tuner as kt
from larq.layers import QuantConv1D, QuantDense
from larq.models import summary
import larq_compute_engine as lce
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
import numpy as np
import scipy.io
import joblib
import numpy as np
import pandas as pd
from tabulate import tabulate

In [ ]:
# Check if GPU is Available
print("Num GPUs Available: ", tf.config.list_physical_devices('GPU'))
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    # Print information about each GPU
    for gpu in gpus:
        print("GPU:", gpu.name)
else:
    print("No GPU devices found.")

## Dataset Preparation

In [ ]:
if not os.path.exists("AsphaltRegularity"):
    print("Downloading AsphaltRegularity dataset...")
    !wget https://www.timeseriesclassification.com/aeon-toolkit/AsphaltRegularity.zip
    !unzip AsphaltRegularity.zip
    !rm AsphaltRegularity.zip
else:
    print("AsphaltRegularity dataset already exists.")


In [ ]:
def load_ts_file(filepath):
    """Load time series data from .ts file"""
    with open(filepath, 'r') as file:
        lines = file.readlines()
        
    # Skip header lines
    data_start_idx = 0
    for i, line in enumerate(lines):
        if line.strip() == '@data':
            data_start_idx = i + 1
            break
    
    X = []
    y = []
    
    # Parse data lines
    for line in lines[data_start_idx:]:
        try:
            # Split line into data and label
            parts = line.strip().split(',')
            
            # Convert time series data to float
            values = [float(val) for val in parts[:-1]]
            X.append(np.array(values, dtype=np.float32))
            
            # Parse label (regular = 0, deteriorated = 1)
            # Format is "number:label", we only want the label part
            label_str = parts[-1].strip().split(':')[-1].lower()
            if label_str == 'regular':
                label = 0
            elif label_str == 'deteriorated':
                label = 1
            else:
                print(f"Unknown label found: {label_str}")
                continue
                
            y.append(label)
            
        except Exception as e:
            print(f"Error parsing line: {line}")
            print(f"Error: {e}")
            continue
    
    X = np.array(X)
    y = np.array(y, dtype=np.int32)
    
    print(f"File {filepath}:")
    print(f"Number of samples: {len(X)}")
    print(f"Unique labels: {np.unique(y)}")
    print(f"Label distribution: {np.bincount(y)}\n")
    
    return X, y

In [ ]:
class AsphaltDataLoader:
    def __init__(self, data_dir='AsphaltRegularity', seed=42):
        self.data_dir = data_dir
        self.seed = seed
        self.scaler = StandardScaler()
        
    def load_dataset(self, batch_size=32, val_split=0.15):
        # Load training data from _eq file
        train_file = os.path.join(self.data_dir, 'AsphaltRegularity_eq_TRAIN.ts')
        X_train_raw, y_train = load_ts_file(train_file)
        
        # Load test data from _eq file
        test_file = os.path.join(self.data_dir, 'AsphaltRegularity_eq_TEST.ts')
        X_test_raw, y_test = load_ts_file(test_file)
        
        # Print dataset statistics
        print("Training samples:", len(X_train_raw))
        print("Test samples:", len(X_test_raw))
        print("Unique classes in training:", np.unique(y_train))
        print("Class distribution in training:", np.bincount(y_train))
        
        # Scale the features
        X_train = self.scaler.fit_transform(X_train_raw)
        X_test = self.scaler.transform(X_test_raw)
        
        # Split training data into train and validation
        train_size = int(len(X_train) * (1 - val_split))
        
        # Shuffle indices
        indices = np.arange(len(X_train))
        np.random.seed(self.seed)
        np.random.shuffle(indices)
        
        # Create train and validation sets
        X_train_final = X_train[indices[:train_size]]
        y_train_final = y_train[indices[:train_size]]
        X_val = X_train[indices[train_size:]]
        y_val = y_train[indices[train_size:]]
        
        # Reshape data to add channel dimension
        X_train_final = X_train_final.reshape(-1, X_train_final.shape[1], 1)
        X_val = X_val.reshape(-1, X_val.shape[1], 1)
        X_test = X_test.reshape(-1, X_test.shape[1], 1)
        
        # Create TensorFlow datasets
        train_ds = tf.data.Dataset.from_tensor_slices((X_train_final, y_train_final))\
            .shuffle(buffer_size=10000)\
            .batch(batch_size)\
            .prefetch(tf.data.AUTOTUNE)
            
        val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))\
            .batch(batch_size)\
            .prefetch(tf.data.AUTOTUNE)
            
        test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))\
            .batch(batch_size)\
            .prefetch(tf.data.AUTOTUNE)
        
        sequence_length = X_train_final.shape[1]
        
        return train_ds, val_ds, test_ds, sequence_length

In [ ]:
def load_asphalt_dataset(batch_size=32, seed=42):
    loader = AsphaltDataLoader(seed=seed)
    return loader.load_dataset(batch_size=batch_size)

In [ ]:
train_ds, val_ds, test_ds, sequence_length = load_asphalt_dataset(batch_size=32)

In [ ]:
for x, y in train_ds.take(1):
    print("\nDataset shapes:")
    print("Input shape:", x.shape)
    print("Label shape:", y.shape)
    print("Unique classes:", len(tf.unique(y)[0]))
    print("Sequence length:", sequence_length)

## NN Evaluation Methods

In [ ]:
def evaluate_NN_models(model_list, test_ds, bin_class=False):
    loss, accuracy, y_pred, precision, recall, f1_score, support, confusion_matrix_list = [], [], [], [], [], [], [], []
    
    # Get all labels from the dataset
    y_test = np.concatenate([y for x, y in test_ds], axis=0)
    
    for model in model_list:
        loss_value, accuracy_value = model.evaluate(test_ds, verbose=0)
        loss.append(loss_value)
        accuracy.append(accuracy_value)
        
        if bin_class:
            y_pred_value = (model.predict(test_ds) > 0.5).astype(int)
        else:
            y_pred_value = np.argmax(np.vstack(model.predict(test_ds)), axis=1)
            
        y_pred.append(y_pred_value)
        y_test_classes = y_test
        
        precision_value, recall_value, f1_score_value, support_value = precision_recall_fscore_support(
            y_test_classes, 
            y_pred_value, 
            average='binary' if bin_class else 'weighted',
            zero_division=0
        )
        
        precision.append(precision_value)
        recall.append(recall_value)
        f1_score.append(f1_score_value)
        support.append(support_value)
        
        confusion_matrix_value = confusion_matrix(y_test_classes, y_pred_value)
        confusion_matrix_list.append(confusion_matrix_value)
        
    return [accuracy, precision, recall, f1_score], confusion_matrix_list, y_pred

In [ ]:
def print_table(data,headers):
    table_data = list(zip(*data))
    table = tabulate(table_data, headers=headers, tablefmt='grid')
    print(table)

In [ ]:
def plot_confusion_matrix(ax, conf_matrix, title, cmap):
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap=cmap, cbar=False, ax=ax)
    ax.set_title(title)

In [ ]:
def draw_confusion_matrix(matrix,name):
    colormap=["Blues","Greens","Oranges"]
    fig, axes = plt.subplots(1, 1, figsize=(15, 5))
    for i in range(len(matrix)):
        plot_confusion_matrix(axes, matrix[i], "Confusion Matrix  ", colormap[i])
    plt.tight_layout()

## Hyperparameters Optimization

In [ ]:
class BinaryNetHyperModel(kt.HyperModel):
    def __init__(self, input_shape, num_classes):
        self.input_shape = input_shape
        self.num_classes = num_classes

    def build(self, hp):
        kwargs = dict(input_quantizer="ste_sign",
                     kernel_quantizer="ste_sign",
                     kernel_constraint="weight_clip")

        model = Sequential()
        
        # Tune the first Conv1D layer
        filters_conv1 = hp.Int('conv1_filters', min_value=16, max_value=64, step=16)
        model.add(Conv1D(filters=filters_conv1, kernel_size=3, activation='relu', 
                        input_shape=self.input_shape))
        model.add(MaxPooling1D(pool_size=2))
        model.add(BatchNormalization(scale=False))

        # Tune the first QuantConv1D layer
        filters_qconv1 = hp.Int('qconv1_filters', min_value=4, max_value=16, step=4)
        model.add(QuantConv1D(filters_qconv1, kernel_size=3,
                             kernel_quantizer="ste_sign",
                             kernel_constraint="weight_clip",
                             use_bias=False))
        model.add(MaxPooling1D(pool_size=2))
        model.add(BatchNormalization(scale=False))
        model.add(Dropout(rate=hp.Float('dropout1', min_value=0.2, max_value=0.5)))

        # Tune the second QuantConv1D layer
        filters_qconv2 = hp.Int('qconv2_filters', min_value=4, max_value=16, step=4)
        model.add(QuantConv1D(filters_qconv2, kernel_size=3, use_bias=False, **kwargs))
        model.add(MaxPooling1D(pool_size=2))
        model.add(BatchNormalization(scale=False))
        model.add(Dropout(rate=hp.Float('dropout2', min_value=0.2, max_value=0.5)))

        model.add(Flatten())

        # Tune the QuantDense layer
        units_qdense = hp.Int('qdense_units', min_value=4, max_value=16, step=4)
        model.add(QuantDense(units_qdense, use_bias=False, **kwargs))
        model.add(BatchNormalization(scale=False))
        model.add(Dense(1, activation='sigmoid'))

        # Tune learning rate
        learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')
        model.compile(
            optimizer=Adam(learning_rate=learning_rate),
            loss='binary_crossentropy',
            
            metrics=['accuracy']
        )
        return model

In [ ]:
# Initialize the hypermodel
hypermodel = BinaryNetHyperModel(input_shape=(sequence_length, 1), num_classes=2)

# Create the tuner
tuner = kt.BayesianOptimization(
    hypermodel,
    objective='val_accuracy',
    max_trials=20,
    directory='bayesian_opt',
    project_name='asphalt_tune'
)

# Define early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=20,
    verbose=1,
    mode='min',
    restore_best_weights=True
)

# Search for the best hyperparameters
tuner.search(
    train_ds,
    epochs=100,
    batch_size=64,
    validation_data=val_ds,
    callbacks=[early_stopping]
)

# Get the best hyperparameters
best_hps_list = tuner.get_best_hyperparameters(num_trials=3)
best_models_list = tuner.get_best_models(num_models=3)

In [ ]:
# Show the top 3 models
print("\nTop 3 Models Performance:")
print("-" * 50)

for idx, hps in enumerate(best_hps_list, 1):
    print(f"\nModel {idx}:")
    print("Hyperparameters:")
    for param, value in hps.values.items():
        print(f"{param}: {value}")
    print(f"Validation Accuracy: {best_models_list[idx-1].evaluate(val_ds, verbose=0)[1]:.3f}")
        
# Let user choose the model
while True:
    try:
        choice = int(input("\nChoose a model (1, 2, or 3): "))
        if 1 <= choice <= 3:
            break
        print("Please enter 1, 2, or 3")
    except ValueError:
        print("Please enter a valid number")

# Use the chosen model
chosen_hps = best_hps_list[choice - 1]

print(f"\nSelected Model {choice}")
print("Final hyperparameters:")
for param, value in chosen_hps.values.items():
    print(f"{param}: {value}")

## Build BNN

In [ ]:
# All quantized layers except the first will use the same options
kwargs = dict(input_quantizer="ste_sign",
            kernel_quantizer="ste_sign",
            kernel_constraint="weight_clip")

model = Sequential()
# First Conv1D layer with tuned filters
model.add(Conv1D(filters=chosen_hps.get('conv1_filters'), kernel_size=3, activation='relu', 
                 input_shape=(sequence_length,1)))
model.add(MaxPooling1D(pool_size=2))
model.add(BatchNormalization(scale=False))

# First QuantConv1D layer with tuned filters
model.add(QuantConv1D(chosen_hps.get('qconv1_filters'), kernel_size=3,
                      kernel_quantizer="ste_sign",
                      kernel_constraint="weight_clip",
                      use_bias=False))
model.add(MaxPooling1D(pool_size=2))
model.add(BatchNormalization(scale=False))
model.add(Dropout(rate=chosen_hps.get('dropout1')))

# Second QuantConv1D layer with tuned filters
model.add(QuantConv1D(chosen_hps.get('qconv2_filters'), kernel_size=3, use_bias=False, **kwargs))
model.add(MaxPooling1D(pool_size=2))
model.add(BatchNormalization(scale=False))
model.add(Dropout(rate=chosen_hps.get('dropout2')))

model.add(Flatten())

# QuantDense layer with tuned units
model.add(QuantDense(chosen_hps.get('qdense_units'), use_bias=False, **kwargs))
model.add(BatchNormalization(scale=False))
model.add(Dense(1, activation='sigmoid'))

In [ ]:
lr = chosen_hps.get('learning_rate')
epochs = 300
batch_size = 64

In [ ]:
model.compile(optimizer=Adam(learning_rate=lr), loss='binary_crossentropy', metrics=['accuracy'])
print(summary(model))

### Train

In [ ]:
history= model.fit(train_ds, epochs=epochs, batch_size=batch_size, validation_data=val_ds, callbacks=[early_stopping], verbose=1)

## Evaluate

In [ ]:
result_all_models_FC, confusion_matrix_FC, predicitions_FC = evaluate_NN_models([model], test_ds, bin_class=True)

In [ ]:
headers=['models','accuracy','precision','recall','f1_score']
data_normal_FC=[["model"]]
data_normal_FC.extend(result_all_models_FC)
print_table(data_normal_FC,headers)

In [ ]:
draw_confusion_matrix(confusion_matrix_FC,"Confusion Matrix")

In [ ]:
plt.figure()
plt.plot(history.history['val_accuracy'], color='orange', label='Validation Accuracy')
plt.xlabel("epochs")
plt.ylabel("accuracy")
plt.show()

## Save

In [ ]:
# Convert our Keras model to a TFLite flatbuffer file
with open("AR_model.tflite", "wb") as flatbuffer_file:
    flatbuffer_bytes = lce.convert_keras_model(model)
    flatbuffer_file.write(flatbuffer_bytes)

## Explainability - SHAP Analysis

In [ ]:
import sys
sys.path.insert(0, '.')
from explainability_analysis import run_multi


X_test = np.vstack([x.numpy() for x, _ in test_ds])
y_test = np.concatenate([y.numpy() for _, y in test_ds])

os.makedirs('inputs/AR', exist_ok=True)
np.save('inputs/AR/AR_test_X.npy', X_test)
np.save('inputs/AR/AR_test_y.npy', y_test)

sample_indices = list(range(min(5, len(X_test))))
run_multi(
    data_dir='inputs',
    model_dir='.',
    out_dir='explainability_output/AR',
    task='AR',
    indices=sample_indices,
    n_shap_bg=30
)

print(f"\n✓ SHAP analysis completed for AR samples {sample_indices}")
print(f"  Outputs saved to: explainability_output/AR/")


## Convert Model to C using CBin-NN

In [ ]:
# Save the trained Keras model to .h5
MODEL_H5_NAME = "AR_model.h5"
model.save(MODEL_H5_NAME)
print(f"Saved Keras model to {MODEL_H5_NAME}")

In [ ]:
# Clone the CBin-NN repo
if not os.path.exists("CBin-NN"):
    !git clone https://github.com/Edge-Learning-Machine/CBin-NN.git
else:
    print("CBin-NN already cloned.")

In [ ]:
# Copy the trained Keras model into the CBin-NN Code Generation folder,
# where model_converter.py expects to find it
import shutil

CODE_GEN_DIR = os.path.join("CBin-NN", "Code Generation")
shutil.copy(MODEL_H5_NAME, os.path.join(CODE_GEN_DIR, MODEL_H5_NAME))
print(f"Copied {MODEL_H5_NAME} to {CODE_GEN_DIR}")

In [ ]:
# Update model_converter.py so model_name points to our trained model
converter_path = os.path.join(CODE_GEN_DIR, "model_converter.py")

with open(converter_path, "r") as f:
    converter_src = f.read()

converter_src = converter_src.replace(
    "model_name = 'lenet_PReLU.h5'",
    f"model_name = '{MODEL_H5_NAME}'"
)

with open(converter_path, "w") as f:
    f.write(converter_src)

print(f"Updated model_name to '{MODEL_H5_NAME}' in {converter_path}")

In [ ]:
# Run model_converter.py from inside the Code Generation folder to generate the C code
!cd "{CODE_GEN_DIR}" && python model_converter.py